# S4 J1 — Architecture d'un mini-framework d'agents

Notebook étudiant généré depuis le Markdown source.

# Objectifs d'apprentissage

## Objectifs principaux

À la fin de ce jour, l'apprenant saura :

1. identifier les composants essentiels d'un framework d'agents IA ;
2. distinguer architecture de framework, architecture d'agent et architecture applicative ;
3. concevoir des interfaces stables avant d'implémenter les détails ;
4. formaliser des dépendances entre composants ;
5. détecter les dépendances circulaires dans une architecture ;
6. documenter les décisions techniques sous forme d'Architecture Decision Records simplifiés ;
7. produire un diagramme Mermaid exploitable dans une documentation technique ;
8. construire un module Python testable qui représente une architecture de framework ;
9. préparer l'évolution vers les jours suivants : Agent abstraction, Tool Registry, Memory Layer, Workflow Engine, Observabilité et Intégration.

## Compétences AI Engineering visées

- Penser en contrats plutôt qu'en scripts.
- Isoler l'orchestration du fournisseur de modèle.
- Éviter le couplage entre outil, mémoire et logique agentique.
- Rendre les décisions techniques auditables.
- Préparer l'observabilité dès l'architecture.
- Construire progressivement un framework au lieu d'empiler des fonctions.

## Ce qui n'est pas l'objectif du jour

Ce jour ne cherche pas à :

- appeler un vrai modèle ;
- construire un runner complet ;
- implémenter une mémoire persistante ;
- reproduire un framework du marché ;
- introduire un moteur multi-agent complet.

Ces éléments arrivent plus tard dans la semaine.

## Chapitre


# Chapitre — Architecture d'un mini-framework d'agents

## 1. Pourquoi commencer par l'architecture ?

Un agent IA peut commencer comme un script :

```python
response = model(prompt)
if response.tool_call:
    result = call_tool(response.tool_call)
```

Ce script fonctionne pour une démo. Il ne suffit pas pour un système maintenable.

Dès que le produit grandit, plusieurs questions apparaissent :

- où enregistrer les outils ?
- qui valide les arguments ?
- qui décide si la boucle continue ?
- où stocker l'état ?
- comment tracer une exécution ?
- comment tester sans appeler un vrai modèle ?
- comment changer de fournisseur de modèle ?
- comment empêcher un outil sensible d'être appelé sans validation ?

L'architecture répond à ces questions avant que le code ne se rigidifie.

## 2. Le problème du script agentique monolithique

Un anti-pattern fréquent consiste à placer dans une seule classe :

- le prompt système ;
- l'appel au modèle ;
- les outils ;
- la mémoire ;
- la boucle ;
- les logs ;
- les règles de sécurité ;
- la logique métier.

Cette classe devient difficile à tester, difficile à relire et difficile à faire évoluer.

```mermaid
flowchart TD
    User[Utilisateur] --> BigAgent[Classe Agent monolithique]
    BigAgent --> Model[Modèle]
    BigAgent --> Tools[Outils]
    BigAgent --> Memory[Mémoire]
    BigAgent --> Logs[Logs]
    BigAgent --> Safety[Règles sécurité]
```

Le problème n'est pas que cette approche ne marche pas. Le problème est qu'elle mélange des responsabilités qui évoluent à des rythmes différents.

## 3. Composants minimaux du framework

Un mini-framework d'agents doit démarrer avec un nombre limité de composants, mais ces composants doivent être bien séparés.

### 3.1 AgentDefinition

`AgentDefinition` décrit ce qu'est un agent.

Il contient par exemple :

- un nom ;
- des instructions ;
- une liste d'outils autorisés ;
- une politique de mémoire ;
- un format de sortie attendu ;
- des garde-fous.

Il ne doit pas exécuter la boucle lui-même.

### 3.2 Runner

`Runner` exécute un agent.

Il orchestre :

- l'entrée utilisateur ;
- les appels au modèle ;
- les appels d'outils ;
- la mise à jour de l'état ;
- la décision d'arrêt ;
- la collecte de traces.

Le `Runner` est le moteur d'exécution.

### 3.3 ModelClient

`ModelClient` encapsule le fournisseur de modèle.

Le framework ne doit pas dépendre directement d'un SDK précis partout dans le code. Il doit définir un port :

```python
class ModelClient:
    def generate(self, messages, tools, output_schema):
        ...
```

L'implémentation peut ensuite utiliser un fournisseur donné.

### 3.4 ToolRegistry

`ToolRegistry` centralise les outils.

Il doit gérer :

- le nom des outils ;
- la description ;
- le schéma d'entrée ;
- la fonction exécutable ;
- les métadonnées de sécurité ;
- les règles d'autorisation.

Il évite que chaque agent reconstruise sa propre liste d'outils.

### 3.5 MemoryStore

`MemoryStore` isole la mémoire.

Il peut stocker :

- l'historique court terme ;
- l'état de tâche ;
- les préférences utilisateur ;
- des résumés ;
- des références externes.

La mémoire ne doit pas être un dictionnaire global non contrôlé.

### 3.6 WorkflowEngine

`WorkflowEngine` structure les étapes.

Il peut représenter :

- une boucle simple ;
- un graphe ;
- des transitions conditionnelles ;
- des limites d'itérations ;
- des handoffs ;
- des validations humaines.

Le jour 5 de la semaine construira ce composant plus en détail.

### 3.7 Observability

`Observability` capture les traces.

Un framework d'agents sans observabilité est difficile à exploiter. Il faut savoir :

- quel prompt a été envoyé ;
- quels outils ont été proposés ;
- quels outils ont été appelés ;
- quelles erreurs sont survenues ;
- combien d'étapes ont été exécutées ;
- pourquoi la boucle s'est arrêtée.

## 4. Frontières et dépendances

L'architecture recommandée suit une dépendance principale :

```text
Application
  -> Runner
    -> AgentDefinition
    -> ModelClient
    -> ToolRegistry
    -> MemoryStore
    -> WorkflowEngine
    -> Observability
```

Une règle importante : les composants bas niveau ne doivent pas dépendre du runner.

Par exemple :

- `ToolRegistry` ne doit pas connaître `Runner`.
- `MemoryStore` ne doit pas connaître `ModelClient`.
- `AgentDefinition` ne doit pas appeler directement un outil.
- `Observability` ne doit pas modifier le résultat métier.

## 5. Invariants d'architecture

Un invariant est une règle que l'architecture doit toujours respecter.

Pour ce mini-framework :

1. un agent est une définition, pas un moteur ;
2. le runner orchestre, mais ne contient pas de logique métier ;
3. le client modèle est remplaçable ;
4. les outils sont déclarés dans un registre ;
5. la mémoire est isolée derrière une interface ;
6. les traces ne modifient jamais l'exécution ;
7. les dépendances circulaires sont interdites ;
8. les garde-fous sont explicites ;
9. les erreurs d'outils sont représentées, pas cachées ;
10. la sérialisation JSON doit être possible pour l'audit.

## 6. Architecture cible de la semaine

```mermaid
flowchart TD
    App[Application] --> Runner[Runner]
    Runner --> Agent[AgentDefinition]
    Runner --> Model[ModelClient]
    Runner --> Tools[ToolRegistry]
    Runner --> Memory[MemoryStore]
    Runner --> Workflow[WorkflowEngine]
    Runner --> Obs[Observability]
    Tools --> ToolSpec[ToolSpec]
    Memory --> State[StateSnapshot]
    Workflow --> Policy[ExecutionPolicy]
    Obs --> Trace[TraceEvent]
```

Cette architecture sera construite progressivement :

- Jour 1 : architecture ;
- Jour 2 : abstraction Agent ;
- Jour 3 : Tool Registry ;
- Jour 4 : Memory Layer ;
- Jour 5 : Workflow Engine ;
- Jour 6 : Observabilité ;
- Jour 7 : Intégration.

## 7. Décisions d'architecture

Une décision d'architecture doit expliquer le choix, pas seulement le résultat.

Exemple :

```text
Décision : AgentDefinition ne contient pas la boucle d'exécution.
Raison : un agent doit rester déclaratif et testable.
Conséquence : le Runner porte la responsabilité d'orchestration.
Tradeoff : il faut définir un contrat clair entre AgentDefinition et Runner.
```

Ce format simple suffit pour ce bootcamp.

## 8. Exemple concret : assistant support

Supposons un assistant support capable de :

- classifier une demande ;
- rechercher une politique interne ;
- proposer une réponse ;
- escalader si le sujet est sensible.

Une mauvaise architecture placerait tout dans une fonction `handle_ticket`.

Une meilleure architecture :

- `AgentDefinition` décrit l'assistant support ;
- `ToolRegistry` expose `search_policy` et `create_ticket`;
- `MemoryStore` garde l'état de la demande ;
- `WorkflowEngine` impose une validation avant escalade ;
- `Observability` trace les décisions ;
- `Runner` orchestre la boucle.

Le gain n'est pas théorique : cette séparation permet de tester l'outil de recherche sans modèle, le runner sans API externe, et les garde-fous sans base de données.

## 9. Critères d'une architecture saine

Une architecture saine permet de répondre simplement à ces questions :

- Puis-je tester une boucle agentique sans appeler le modèle ?
- Puis-je ajouter un outil sans modifier le runner ?
- Puis-je changer de modèle sans modifier les agents ?
- Puis-je inspecter une exécution après coup ?
- Puis-je limiter les permissions d'un outil ?
- Puis-je réutiliser la mémoire dans plusieurs workflows ?
- Puis-je expliquer pourquoi une décision a été prise ?

Si la réponse est non, l'architecture est trop couplée.

## 10. Transition vers le jour 2

Le jour 2 implémentera l'abstraction `Agent`. Le travail du jour 1 garantit que cette abstraction ne sera pas une classe omnisciente.

Le bon objectif du jour 2 sera :

```text
Définir ce qu'un agent est, pas tout ce qu'un agent fait.
```

## Exercices


# Exercices — Architecture d'un mini-framework d'agents

## Exercice 1 — Identifier les responsabilités

Pour chaque responsabilité, indique le composant le plus approprié :

1. Stocker la description d'un outil.
2. Décider si une boucle doit continuer.
3. Envoyer une requête au modèle.
4. Conserver un résumé de conversation.
5. Écrire une trace d'exécution.
6. Décrire les instructions d'un agent.
7. Appliquer une limite d'itérations.
8. Refuser un appel d'outil sensible sans approbation.

## Exercice 2 — Détecter le couplage

Analyse cette pseudo-architecture :

```text
SupportAgent
├── openai_client
├── tools
├── user_profile
├── logs
├── run_loop()
├── save_memory()
├── call_billing_api()
└── validate_security()
```

Réponds :

1. Quelles responsabilités sont mélangées ?
2. Quels composants devraient être extraits ?
3. Quel risque apparaît si on veut ajouter un deuxième agent ?

## Exercice 3 — Écrire un invariant

Écris trois invariants d'architecture pour un framework d'agents.

Chaque invariant doit être formulé comme une règle vérifiable.

Exemple :

```text
Le ToolRegistry ne dépend jamais du Runner.
```

## Exercice 4 — ADR simplifié

Écris un mini Architecture Decision Record pour la décision suivante :

```text
Les outils seront déclarés dans un registre centralisé.
```

Le format attendu :

```text
Décision :
Contexte :
Raison :
Conséquence :
Tradeoff :
```

## Exercice 5 — Diagramme

Complète le diagramme Mermaid suivant avec les composants manquants :

```mermaid
flowchart TD
    App[Application] --> Runner[Runner]
    Runner --> Agent[?]
    Runner --> Model[?]
    Runner --> Tools[?]
    Runner --> Memory[?]
    Runner --> Obs[?]
```

## Exercice 6 — Lecture de code

Dans le lab, exécute :

```bash
python mini_framework_architecture.py
```

Puis réponds :

1. Quels composants sont déclarés ?
2. Quels composants dépendent directement du `runner` ?
3. Le framework autorise-t-il les dépendances circulaires ?
4. Quelle méthode rend le diagramme Mermaid ?

## Challenge


# Challenge — Concevoir l'architecture d'un assistant IA mono-framework

## Contexte

Tu dois concevoir le socle d'un mini-framework destiné à construire un assistant IA pour une équipe support.

L'assistant devra plus tard :

- comprendre une demande utilisateur ;
- appeler des outils internes ;
- conserver un état de tâche ;
- escalader les cas sensibles ;
- tracer chaque étape ;
- être testable sans appel modèle réel.

## Mission

Produis une proposition d'architecture composée de :

1. une liste de composants ;
2. la responsabilité de chaque composant ;
3. les dépendances entre composants ;
4. trois invariants ;
5. deux décisions d'architecture ;
6. un diagramme Mermaid ;
7. un plan d'implémentation en quatre étapes.

## Contraintes

- Aucun composant ne doit tout faire.
- Le modèle doit être remplaçable.
- Les outils doivent être déclarés hors du runner.
- L'observabilité ne doit pas modifier l'exécution.
- La mémoire ne doit pas être un dictionnaire global.
- Les actions sensibles doivent être explicitement représentées.

## Critères de réussite

Le challenge est réussi si :

- l'architecture est compréhensible sans lire le code ;
- les dépendances sont cohérentes ;
- les responsabilités ne se chevauchent pas excessivement ;
- les invariants sont vérifiables ;
- le diagramme reflète le texte ;
- le plan prépare les jours 2 à 7.

## Lab — code de référence sans corrigés

In [ ]:
from dataclasses import dataclass, field
from typing import Any
import json

REQUIRED_COMPONENTS = {
    "agent_definition",
    "runner",
    "model_client",
    "tool_registry",
    "memory_store",
    "workflow_engine",
    "observability",
}

@dataclass(frozen=True)
class ComponentSpec:
    name: str
    responsibility: str
    interfaces: tuple[str, ...] = field(default_factory=tuple)
    depends_on: tuple[str, ...] = field(default_factory=tuple)
    risks: tuple[str, ...] = field(default_factory=tuple)

    def validate(self) -> list[str]:
        errors = []
        if not self.name.strip():
            errors.append("component.name is required")
        if not self.responsibility.strip():
            errors.append(f"{self.name}: responsibility is required")
        if not self.interfaces:
            errors.append(f"{self.name}: at least one interface is required")
        if self.name in self.depends_on:
            errors.append(f"{self.name}: component cannot depend on itself")
        return errors

In [ ]:
# Exercice guidé :
# Crée un ComponentSpec pour un ToolRegistry, puis appelle validate().
tool_registry = ComponentSpec(
    name="tool_registry",
    responsibility="Déclare et valide les outils disponibles.",
    interfaces=("register", "call"),
)
tool_registry.validate()

# Références

## Documentation technique

- OpenAI Agents SDK — Concepts d'agents, runner, tools, handoffs, sessions, guardrails et tracing.
- OpenAI Agents SDK — Tracing pour visualiser, déboguer et monitorer les workflows agentiques.
- Model Context Protocol — Spécifications autour des tools, resources et prompts exposés à un client.
- Architecture hexagonale / Ports and Adapters — séparation entre domaine et intégrations externes.
- Architecture Decision Records — formalisation légère des décisions techniques.

## Pour approfondir

- Séparer la définition d'un agent de son exécution.
- Concevoir une interface de modèle remplaçable.
- Définir un registre d'outils testable.
- Prévoir l'observabilité avant la production.
- Tester une architecture par invariants plutôt que par appels externes.

## À retenir

Une architecture de framework d'agents est réussie lorsqu'elle rend les extensions simples et les erreurs visibles.

# Corrigés formateur

# Corrigé — Exercices

## Exercice 1 — Identifier les responsabilités

1. Stocker la description d'un outil : `ToolRegistry`.
2. Décider si une boucle doit continuer : `WorkflowEngine` ou politique d'exécution utilisée par le `Runner`.
3. Envoyer une requête au modèle : `ModelClient`.
4. Conserver un résumé de conversation : `MemoryStore`.
5. Écrire une trace d'exécution : `Observability`.
6. Décrire les instructions d'un agent : `AgentDefinition`.
7. Appliquer une limite d'itérations : `WorkflowEngine` ou `ExecutionPolicy`.
8. Refuser un appel d'outil sensible sans approbation : `Guardrail` ou métadonnée de sécurité du `ToolRegistry`, appliquée par le `Runner`.

## Exercice 2 — Détecter le couplage

La pseudo-architecture mélange :

- définition de l'agent ;
- appel modèle ;
- stockage mémoire ;
- logique d'outil ;
- logs ;
- sécurité ;
- boucle d'exécution ;
- logique métier support.

Composants à extraire :

- `AgentDefinition`;
- `ModelClient`;
- `ToolRegistry`;
- `MemoryStore`;
- `Runner`;
- `Observability`;
- `Guardrail`;
- éventuellement `WorkflowEngine`.

Le risque principal avec un deuxième agent est la duplication. Chaque nouvel agent réimplémente ses outils, sa mémoire, ses logs et ses validations. Le framework devient incohérent.

## Exercice 3 — Invariants possibles

Exemples :

```text
Le ToolRegistry ne dépend jamais du Runner.
Le ModelClient ne connaît aucun outil métier.
L'Observability ne modifie jamais l'état d'exécution.
La MemoryStore ne déclenche jamais d'appel modèle.
Le Runner ne contient pas de logique métier spécifique à un domaine.
```

## Exercice 4 — ADR simplifié

```text
Décision :
Les outils seront déclarés dans un registre centralisé.

Contexte :
Plusieurs agents devront utiliser des outils communs avec des schémas, permissions et descriptions cohérents.

Raison :
Un registre évite la duplication, rend les outils testables et permet d'appliquer des règles de sécurité uniformes.

Conséquence :
Les agents référencent des outils par nom au lieu de les instancier directement.

Tradeoff :
Il faut maintenir un contrat de registre et gérer les erreurs lorsqu'un outil demandé n'existe pas.
```

## Exercice 5 — Diagramme complété

```mermaid
flowchart TD
    App[Application] --> Runner[Runner]
    Runner --> Agent[AgentDefinition]
    Runner --> Model[ModelClient]
    Runner --> Tools[ToolRegistry]
    Runner --> Memory[MemoryStore]
    Runner --> Obs[Observability]
```

## Exercice 6 — Lecture de code

1. Les composants déclarés sont notamment `application`, `runner`, `agent_definition`, `model_client`, `tool_registry`, `memory_store`, `workflow_engine`, `observability` et `guardrails`.
2. Dans l'architecture cible, aucun composant ne doit dépendre du `runner` sauf l'application qui l'utilise.
3. Non. Le validateur refuse les dépendances circulaires.
4. La méthode `render_mermaid()` rend le diagramme Mermaid.

# Corrigé — Interview

## Question 1

Une classe `Agent` qui fait tout mélange définition, exécution, outils, mémoire, sécurité et observabilité. Elle devient difficile à tester, à remplacer et à étendre. Chaque évolution touche le même bloc de code, ce qui augmente le risque de régression.

## Question 2

`AgentDefinition` décrit un agent : nom, instructions, outils autorisés, format de sortie, politiques. `Runner` exécute un agent : il gère les tours, les appels modèle, les appels outils, l'état, les traces et les conditions d'arrêt.

## Question 3

Le `ToolRegistry` doit être séparé pour éviter que chaque runner ou agent redéclare les outils. Cela permet aussi de centraliser validation, documentation, permissions et tests.

## Question 4

Une interface `ModelClient` rend le fournisseur de modèle remplaçable. Le framework peut être testé avec un faux client et évoluer sans réécrire toute l'orchestration.

## Question 5

Un invariant est une règle d'architecture qui doit toujours rester vraie. Exemple : `Observability` peut lire les événements d'exécution, mais ne doit jamais modifier l'état métier.

## Question 6

Signaux de couplage :

- une classe connaît tous les détails ;
- les outils importent le runner ;
- la mémoire appelle directement le modèle ;
- les tests nécessitent une vraie API ;
- ajouter un agent oblige à copier du code ;
- les logs contiennent de la logique métier.

## Question 7

L'observabilité doit être prévue dès le début, car les agents sont non déterministes et multi-étapes. Sans trace, il devient difficile de comprendre pourquoi un outil a été appelé ou pourquoi une réponse a été produite.

## Question 8

Il faut préparer MCP comme une intégration possible derrière un port. Le framework peut définir un `ToolProvider` ou un adaptateur MCP sans rendre le runner dépendant directement du protocole.

## Question 9

Les limites d'itérations et conditions d'arrêt appartiennent au `WorkflowEngine` ou à une `ExecutionPolicy` utilisée par le runner.

## Question 10

On peut tester avec :

- un faux `ModelClient`;
- des outils déterministes ;
- une mémoire en mémoire ;
- des traces capturées localement ;
- des tests d'invariants d'architecture ;
- des scénarios JSON.

# Corrigé — Challenge

## Proposition d'architecture

### Composants

| Composant | Responsabilité |
|---|---|
| `AgentDefinition` | Décrire l'agent, ses instructions, ses outils autorisés et ses politiques. |
| `Runner` | Exécuter une boucle agentique à partir d'une définition d'agent. |
| `ModelClient` | Encapsuler l'appel au modèle. |
| `ToolRegistry` | Déclarer, valider et exécuter les outils. |
| `MemoryStore` | Lire et écrire l'état de conversation ou de tâche. |
| `WorkflowEngine` | Gérer transitions, limites, conditions d'arrêt et validations humaines. |
| `Observability` | Capturer événements, traces, erreurs et métriques. |
| `Guardrails` | Bloquer ou contrôler les entrées, sorties et appels sensibles. |

### Dépendances

```text
Application -> Runner
Runner -> AgentDefinition
Runner -> ModelClient
Runner -> ToolRegistry
Runner -> MemoryStore
Runner -> WorkflowEngine
Runner -> Observability
Runner -> Guardrails
```

Les composants spécialisés ne dépendent pas du runner.

### Invariants

```text
Le Runner orchestre, mais ne contient pas de logique métier support.
Le ToolRegistry est la seule source de vérité des outils.
L'Observability ne modifie jamais l'état métier.
```

### Décisions d'architecture

#### ADR 1 — Outils centralisés

```text
Décision :
Les outils seront déclarés dans ToolRegistry.

Contexte :
Plusieurs agents pourront utiliser des outils communs.

Raison :
Un registre central réduit la duplication et facilite la sécurité.

Conséquence :
Les agents référencent les outils par nom.

Tradeoff :
Le registre devient un composant critique à tester soigneusement.
```

#### ADR 2 — ModelClient comme port

```text
Décision :
Le modèle sera appelé via une interface ModelClient.

Contexte :
Le framework doit rester testable et remplaçable.

Raison :
Les détails fournisseur ne doivent pas polluer le runner.

Conséquence :
Les tests peuvent utiliser un faux client modèle.

Tradeoff :
Il faut maintenir une couche d'adaptation supplémentaire.
```

### Diagramme Mermaid

```mermaid
flowchart TD
    App[Application support] --> Runner[Runner]
    Runner --> Agent[AgentDefinition]
    Runner --> Model[ModelClient]
    Runner --> Tools[ToolRegistry]
    Runner --> Memory[MemoryStore]
    Runner --> Workflow[WorkflowEngine]
    Runner --> Obs[Observability]
    Runner --> Guardrails[Guardrails]
    Tools --> Search[search_policy]
    Tools --> Ticket[create_ticket]
    Workflow --> Approval[Human approval]
```

### Plan d'implémentation

1. Créer les dataclasses et contrats : `AgentDefinition`, `ToolSpec`, `ExecutionPolicy`.
2. Implémenter un `Runner` minimal avec faux `ModelClient`.
3. Ajouter `ToolRegistry`, validation et erreurs contrôlées.
4. Ajouter mémoire, workflow, observabilité et intégration finale.

## Commentaire formateur

Une bonne réponse évite de placer la logique support dans le runner. Le runner doit pouvoir exécuter plus tard un agent support, un agent recherche ou un agent code sans modification profonde.

# Review formateur — Semaine 4 Jour 1

## Points à vérifier

- L'apprenant distingue définition d'agent et exécution.
- L'apprenant sait expliquer pourquoi le runner ne doit pas contenir toute la logique.
- Les responsabilités sont réparties sans chevauchement majeur.
- Les dépendances ne forment pas de cycle.
- Les invariants sont vérifiables.
- Le diagramme Mermaid reflète l'architecture écrite.
- Le code du lab est exécuté et les tests passent.

## Erreurs fréquentes

### 1. Créer une classe `Agent` omnisciente

Symptôme :

```python
class Agent:
    def call_model(self): ...
    def call_tool(self): ...
    def save_memory(self): ...
    def log(self): ...
    def validate_security(self): ...
```

Correction : distinguer `AgentDefinition`, `Runner`, `ToolRegistry`, `MemoryStore` et `Observability`.

### 2. Faire dépendre les outils du runner

Un outil doit exécuter une action métier ou technique. Il ne doit pas connaître la boucle agentique.

### 3. Oublier les traces

Les traces ne sont pas un détail de production. Elles sont nécessaires dès les tests pour comprendre les décisions de l'agent.

### 4. Confondre mémoire et état

L'état décrit une tâche en cours. La mémoire peut survivre à plusieurs sessions. Les deux doivent rester explicites.

## Questions de validation orale

- Où placerais-tu une validation humaine avant suppression de données ?
- Comment ajouterais-tu un second fournisseur de modèle ?
- Comment testerais-tu le runner sans modèle réel ?
- Quel composant devrait exposer des outils MCP ?
- Que se passe-t-il si deux composants dépendent mutuellement l'un de l'autre ?

## Critère de passage au jour 2

L'apprenant peut passer au jour 2 s'il est capable de dessiner l'architecture du mini-framework et d'expliquer la responsabilité de chaque composant sans utiliser une classe unique qui fait tout.

In [ ]:
# Validation rapide du lab complet depuis le notebook formateur.
import pathlib, runpy, sys
lab_dir = pathlib.Path.cwd()
print("Notebook formateur prêt. Exécuter les tests depuis le dossier labs si nécessaire.")